In [12]:
import os
print("Working dir:", os.getcwd())
print("Contents:", os.listdir("."))

Working dir: c:\Users\Arianna Yuan\Desktop\Academia\USC\Semesters\SPRING 2026\CSCI 567 -- Machine Learning\Project\modma_depression_detection\scripts
Contents: ['.gitattributes', 'analysis.ipynb', 'analysis_intervals_and_groupings.ipynb', 'extract_opensmile_features.py', 'extract_wavlm_features.py', 'extract_whisper_features.py']


In [13]:
# Check if modma_data is visible
print(os.listdir("../modma_data"))

['audio_lanzhou_2015', 'chan_info_egi_128.mat', 'EEG_128channels_ERP_lanzhou_2015', 'EEG_3channels_resting_lanzhou_2015', 'given_papers', 'subjects_information.xlsx', 'The pre-processing of Task-State EEG data.docx']


In [14]:
from pathlib import Path
%pip install pandas
import pandas as pd

AUDIO_ROOT = Path("../modma_data/audio_lanzhou_2015")
DROP_COLS = ("file", "start", "end")  # openSMILE metadata we don't want as features

func_dfs: dict[int, pd.DataFrame] = {}
subject_dirs = sorted(p for p in AUDIO_ROOT.iterdir() if p.is_dir())

for file_idx in range(1, 30):
    stem = f"{file_idx:02d}"
    rows: dict[str, pd.Series] = {}
    feature_cols: list[str] | None = None

    for subj_dir in subject_dirs:
        func_csv = subj_dir / f"{stem}_openSMILE_func.csv"
        if not func_csv.exists():
            continue
        df = pd.read_csv(func_csv)
        df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

        if feature_cols is None:
            feature_cols = list(df.columns)
        elif list(df.columns) != feature_cols:
            raise ValueError(
                f"Column mismatch in {func_csv}:\n"
                f"  expected {feature_cols[:3]}...\n"
                f"  got      {list(df.columns)[:3]}..."
            )

        rows[subj_dir.name] = df.iloc[0]

    out = pd.DataFrame.from_dict(rows, orient="index", columns=feature_cols)
    out.index.name = "subject_id"
    func_dfs[file_idx] = out

# Sanity check — should print real feature names, not 0..87
print(func_dfs[1].columns.tolist()[:5])
print(func_dfs[1].shape)

Note: you may need to restart the kernel to use updated packages.
['F0semitoneFrom27.5Hz_sma3nz_amean', 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'F0semitoneFrom27.5Hz_sma3nz_percentile20.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile50.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile80.0']
(11, 88)


In [15]:
func_dfs[1]

,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
subject_id,,,,,,,,,,,,,,,,,,,,,
02010015,23.860804,0.153021,21.573397,24.685276,26.444733,4.871336,298.551700,331.271330,80.673350,149.378070,...,-0.039258,-0.009936,0.025072,2.485796,1.997147,0.309286,0.365591,0.182308,0.279564,-42.918255
02010034,25.970173,0.136090,24.199432,25.936773,28.924982,4.725550,245.073800,188.476290,65.341060,56.882263,...,-0.055031,-0.001629,0.013199,1.485148,1.420454,0.147000,0.075901,0.534000,0.658380,-53.755997
02010039,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.040194,-0.002549,0.003542,1.226636,0.000000,0.000000,0.000000,17.050000,0.000000,-72.296480
02020007,34.902786,0.144602,33.536000,35.412240,37.838707,4.302708,325.595900,382.158970,32.812110,76.722960,...,-0.043036,-0.008856,0.025354,3.265602,1.749271,0.306667,0.190343,0.233200,0.277427,-48.003555
02020010,20.063597,0.107313,18.050644,19.986654,21.991734,3.941090,157.896620,155.373840,37.359604,42.883045,...,-0.053509,-0.002444,0.021615,2.544769,2.464455,0.168077,0.120799,0.223600,0.308297,-51.343830
02020011,22.874650,0.189830,19.915375,22.008790,24.991901,5.076527,162.551130,197.806870,52.164330,44.368256,...,-0.058731,-0.002253,0.021077,2.915452,1.757812,0.215000,0.124421,0.332778,0.422989,-47.658024
02020015,34.695183,0.042828,34.308760,34.947216,35.324245,1.015484,209.064670,152.932390,25.963501,22.531520,...,-0.059072,-0.007070,0.020400,2.550091,1.654412,0.097778,0.049166,0.394545,0.502310,-56.969020
02020016,22.223309,0.091082,20.178535,22.967680,23.740845,3.562309,42.034092,24.134161,135.252730,190.580630,...,-0.079886,-0.005560,0.030164,1.996672,1.342282,0.131250,0.080845,0.526667,0.594549,-52.574173
02020018,23.971468,0.071886,22.255123,23.863651,25.663550,3.408426,90.167725,101.605290,14.815989,7.482521,...,-0.084798,-0.006416,0.022915,1.480485,1.084011,0.162500,0.068875,0.656667,0.816061,-55.688614


In [18]:
import os
sample = os.listdir("../modma_data/audio_lanzhou_2015")
print("First few subjects:", sample[:3])
# Pick one and list its contents
subj = sample[0]
files = sorted(os.listdir(f"../modma_data/audio_lanzhou_2015/{subj}"))
print(f"\nFiles in {subj}:")
for f in files:
    print(f"  {f}")

First few subjects: ['02010015', '02010023', '02010034']

Files in 02010015:
  01_openSMILE_func.csv
  02_openSMILE_func.csv
  03_openSMILE_func.csv
  04_openSMILE_func.csv
  04_transcript_faster_whisper_zh.txt
  05_transcript_faster_whisper_zh.txt
  06_transcript_faster_whisper_zh.txt
  07_transcript_faster_whisper_zh.txt
  08_openSMILE_func.csv
  08_transcript_faster_whisper_zh.txt
  09_openSMILE_func.csv
  09_openSMILE_lld.csv
  09_transcript_faster_whisper_zh.txt
  10_openSMILE_func.csv
  10_transcript_faster_whisper_zh.txt
  11_openSMILE_func.csv
  11_transcript_faster_whisper_zh.txt
  12_openSMILE_func.csv
  12_transcript_faster_whisper_zh.txt
  13_openSMILE_func.csv
  13_openSMILE_lld.csv
  13_transcript_faster_whisper_zh.txt
  14_openSMILE_func.csv
  14_openSMILE_lld.csv
  14_transcript_faster_whisper_zh.txt
  15_openSMILE_func.csv
  15_openSMILE_lld.csv
  15_transcript_faster_whisper_zh.txt
  16_openSMILE_lld.csv
  16_transcript_faster_whisper_zh.txt
  17_transcript_faster_whi

In [24]:
# CELL 3 — Load the subject-info and audio-file-map CSVs.
#
# subject_info:   one row per audio subject (52 rows).
#                 Columns: group, age, gender, edu_years, PHQ-9,
#                          CTQ-SF, LES, SSRS, GAD-7, PSQI.
#                 Indexed by subject_id (zero-padded 8-digit string).
# audio_file_map: one row per wav file number 1..29 (29 rows).
#                 Columns: task, task_number, valence, notes.

from pathlib import Path
import pandas as pd

CSV_ROOT = Path("../modma_data")

subject_info = pd.read_csv(
    CSV_ROOT / "subject_info_map.csv",
    dtype={"subject_id": str},
).set_index("subject_id")

audio_file_map = pd.read_csv(CSV_ROOT / "audio_file_map.csv")

print(f"subject_info:    {subject_info.shape}  "
      f"groups={subject_info['group'].value_counts().to_dict()}")
print(f"audio_file_map:  {audio_file_map.shape}  "
      f"tasks={audio_file_map['task'].value_counts().to_dict()}")
print(f"PHQ-9 — MDD range {subject_info.loc[subject_info.group=='MDD', 'PHQ-9'].min()}"
      f"–{subject_info.loc[subject_info.group=='MDD', 'PHQ-9'].max()}, "
      f"HC range {subject_info.loc[subject_info.group=='HC', 'PHQ-9'].min()}"
      f"–{subject_info.loc[subject_info.group=='HC', 'PHQ-9'].max()}")
subject_info.head(3)


subject_info:    (52, 10)  groups={'HC': 29, 'MDD': 23}
audio_file_map:  (29, 5)  tasks={'Interview': 18, 'Word Reading': 6, 'Picture Description': 4, 'Passage Reading': 1}
PHQ-9 — MDD range 6–25, HC range 0–9


,group,age,gender,edu_years,PHQ-9,CTQ-SF,LES,SSRS,GAD-7,PSQI
subject_id,,,,,,,,,,
02010002,MDD,18,F,12,23,77,-143,31,18,12
02010004,MDD,25,F,19,12,53,-44,38,13,11
02010005,MDD,20,M,16,19,49,-3,28,11,5


In [25]:
# How many subjects have audio folders?
import os
all_folders = [d for d in os.listdir("../modma_data/audio_lanzhou_2015") if os.path.isdir(f"../modma_data/audio_lanzhou_2015/{d}")]
print(f"Audio folders: {len(all_folders)}")

# How many have openSMILE func CSVs?
has_func = [d for d in all_folders if os.path.exists(f"../modma_data/audio_lanzhou_2015/{d}/01_openSMILE_func.csv")]
print(f"Folders with openSMILE features: {len(has_func)}")

# Which subjects are in subject_info but missing from audio?
missing = set(subject_info.index) - set(subject_mean_88.index)
print(f"Subjects in subject_info but missing audio features: {len(missing)}")

Audio folders: 35
Folders with openSMILE features: 11
Subjects in subject_info but missing audio features: 27


In [26]:
from pathlib import Path
root = Path("../modma_data/audio_lanzhou_2015")
count = 0
for subj in sorted(root.iterdir()):
    if subj.is_dir():
        funcs = list(subj.glob("*_openSMILE_func.csv"))
        if funcs:
            count += 1
            print(f"  {subj.name}: {len(funcs)} func CSVs")
print(f"\nTotal subjects with func CSVs: {count}")

  02010015: 24 func CSVs
  02010034: 1 func CSVs
  02010037: 1 func CSVs
  02010039: 3 func CSVs
  02020004: 2 func CSVs
  02020007: 7 func CSVs
  02020008: 3 func CSVs
  02020010: 6 func CSVs
  02020011: 3 func CSVs
  02020015: 25 func CSVs
  02020016: 7 func CSVs
  02020018: 3 func CSVs
  02020019: 3 func CSVs
  02020023: 12 func CSVs
  02020026: 17 func CSVs
  02020027: 1 func CSVs
  02030001: 12 func CSVs
  02030002: 13 func CSVs
  02030004: 15 func CSVs
  02030005: 1 func CSVs
  02030007: 1 func CSVs
  02030009: 5 func CSVs
  02030014: 2 func CSVs
  02030016: 21 func CSVs
  02030017: 2 func CSVs

Total subjects with func CSVs: 25


In [27]:
from pathlib import Path
root = Path("../modma_data/audio_lanzhou_2015")
for subj in sorted(root.iterdir()):
    if subj.is_dir():
        wavs = list(subj.glob("*.wav"))
        print(f"  {subj.name}: {len(wavs)} wavs")

  02010015: 0 wavs
  02010023: 3 wavs
  02010034: 0 wavs
  02010036: 0 wavs
  02010037: 11 wavs
  02010039: 13 wavs
  02020004: 0 wavs
  02020007: 1 wavs
  02020008: 0 wavs
  02020010: 0 wavs
  02020011: 0 wavs
  02020014: 4 wavs
  02020015: 3 wavs
  02020016: 2 wavs
  02020018: 1 wavs
  02020019: 3 wavs
  02020021: 1 wavs
  02020022: 0 wavs
  02020023: 0 wavs
  02020025: 0 wavs
  02020026: 1 wavs
  02020027: 7 wavs
  02030001: 9 wavs
  02030002: 0 wavs
  02030004: 0 wavs
  02030005: 11 wavs
  02030006: 4 wavs
  02030007: 4 wavs
  02030008: 8 wavs
  02030009: 3 wavs
  02030010: 0 wavs
  02030014: 0 wavs
  02030015: 1 wavs
  02030016: 0 wavs
  02030017: 3 wavs


In [23]:
# CELL 4 — Aggregate per-subject mean across the 29 openSMILE files.
#
# Stack all 29 (52 x 88) DataFrames in func_dfs into a tall frame, then
# group-by subject_id and take the mean. Subjects with missing files
# (e.g. 02010004 is missing 5 corrupt wavs) are averaged over whatever
# files they do have.

stacked = pd.concat(func_dfs.values(), axis=0)
subject_mean_88 = stacked.groupby(level=0).mean()
subject_mean_88.index.name = "subject_id"

# Align to subject_info row order (should already match)
subject_mean_88 = subject_mean_88.loc[subject_info.index]

files_per_subject = stacked.groupby(level=0).size()
print(f"Files per subject — min {files_per_subject.min()}, "
      f"median {int(files_per_subject.median())}, "
      f"max {files_per_subject.max()}")
missing = files_per_subject[files_per_subject < 29]
if len(missing):
    print(f"Subjects with <29 files: {missing.to_dict()}")
else:
    print("All subjects have all 29 files.")

print(f"\nsubject_mean_88 shape: {subject_mean_88.shape}")
subject_mean_88.head(3)


KeyError: "['02010002', '02010004', '02010005', '02010006', '02010008', '02010009', '02010010', '02010011', '02010012', '02010013', '02010018', '02010022', '02010023', '02010024', '02010025', '02010036', '02030006', '02030008', '02030015', '02020014', '02020021', '02020022', '02020025', '02030010', '02010001', '02010003', '02010014'] not in index"

In [ ]:
# CELL 5 — 80/20 train/test split, STRATIFIED by MDD/HC group.
#
# Should we stratify? YES.
#
#  (a) n=52 with 23 MDD / 29 HC (44% / 56%). A random unstratified
#      80/20 split leaves ~10 test subjects; class composition of
#      that test set could range from 1 to 7 MDD purely by chance.
#  (b) Stratified CV/splitting reduces estimate variance and bias
#      at small n (Kohavi 1995, IJCAI, "A Study of Cross-Validation
#      and Bootstrap for Accuracy Estimation and Model Selection").
#  (c) PHQ-9 is bimodal. An unstratified test set could be
#      dominated by one mode, yielding a misleading RMSE that
#      doesn't reflect performance across the target range.
#  (d) Standard practice in clinical-speech depression work
#      (Cummins et al. 2015 Speech Communication review;
#      Low et al. 2020 Laryngoscope Investig. Otolaryngol. review).
#
# Alternative: stratify by binned PHQ-9. Overkill for a 10-point
# test set — group-label stratification covers the bimodality.

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_all = subject_mean_88.values                  # (52, 88)
y_all = subject_info["PHQ-9"].values            # (52,)
groups_all = subject_info["group"].values       # (52,) — MDD or HC
idx_all = subject_info.index.values             # subject_ids

X_train, X_test, y_train, y_test, grp_train, grp_test, idx_train, idx_test = train_test_split(
    X_all, y_all, groups_all, idx_all,
    test_size=0.20, stratify=groups_all, random_state=RANDOM_STATE,
)

print(f"Train: X={X_train.shape}  groups={pd.Series(grp_train).value_counts().to_dict()}  "
      f"PHQ-9 range {y_train.min()}-{y_train.max()}")
print(f"Test : X={X_test.shape}   groups={pd.Series(grp_test).value_counts().to_dict()}  "
      f"PHQ-9 range {y_test.min()}-{y_test.max()}")
print(f"\nTest subject_ids: {list(idx_test)}")


In [ ]:
# CELL 6 — LOSO Elastic Net on the TRAINING set.
#
# Structure:
#  1. Fit StandardScaler on TRAIN ONLY and apply to train+test (no leakage).
#  2. ElasticNetCV with LeaveOneOut inner CV selects (alpha, l1_ratio) on
#     training data. This is the "LOSO" part — each training subject is
#     held out in turn during hyperparameter search.
#  3. ElasticNetCV refits the final model on the full training set at the
#     chosen hyperparameters.
#  4. Separately, generate training LOSO out-of-fold predictions via
#     cross_val_predict for diagnostic plotting in Cell 8.

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.model_selection import LeaveOneOut, cross_val_predict

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

loo = LeaveOneOut()

enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
    alphas=np.logspace(-3, 1, 40),
    cv=loo,
    max_iter=20000,
    n_jobs=-1,
    random_state=RANDOM_STATE,
).fit(X_train_s, y_train)

n_nonzero = int((enet_cv.coef_ != 0).sum())
print(f"ElasticNetCV — chosen hyperparameters:")
print(f"  alpha (lambda) = {enet_cv.alpha_:.4f}")
print(f"  l1_ratio       = {enet_cv.l1_ratio_:.2f}  "
      f"(1.0=pure LASSO, 0.0=pure Ridge)")
print(f"  nonzero coefs  = {n_nonzero} / {len(enet_cv.coef_)}")

# Training LOSO out-of-fold predictions at the chosen hyperparameters
final_enet = ElasticNet(
    alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_,
    max_iter=20000, random_state=RANDOM_STATE,
)
loso_pred_train = cross_val_predict(final_enet, X_train_s, y_train, cv=loo)
print(f"\nloso_pred_train shape: {loso_pred_train.shape}  "
      f"(one out-of-fold prediction per training subject)")


In [ ]:
# CELL 7 — Per-test-subject loss and overall metrics.
# Also compare against training LOSO-OOF performance (checks for overfit
# to the outer split) and a trivial mean-predictor baseline.

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_test = enet_cv.predict(X_test_s)

test_results = pd.DataFrame({
    "subject_id": idx_test,
    "group": grp_test,
    "PHQ9_actual": y_test,
    "PHQ9_pred": y_pred_test,
    "abs_error": np.abs(y_test - y_pred_test),
    "sq_error": (y_test - y_pred_test) ** 2,
}).set_index("subject_id").sort_values("abs_error", ascending=False)

print("Per-test-subject predictions (sorted by abs_error desc):")
print(test_results.round(3))

print("\n--- Test-set metrics ---")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")
print(f"  MAE : {mean_absolute_error(y_test, y_pred_test):.3f}")
print(f"  R^2 : {r2_score(y_test, y_pred_test):.3f}")

print("\n--- Training LOSO-OOF metrics (diagnostic) ---")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, loso_pred_train)):.3f}")
print(f"  MAE : {mean_absolute_error(y_train, loso_pred_train):.3f}")
print(f"  R^2 : {r2_score(y_train, loso_pred_train):.3f}")

baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
print(f"\nMean-predictor baseline test RMSE: {np.sqrt(mean_squared_error(y_test, baseline_pred)):.3f}")
print(f"(If Elastic Net test RMSE >= baseline, the model adds no value.)")


In [ ]:
# CELL 8 — Visualization.
# Left panel:  predicted vs actual PHQ-9. Train points = LOSO out-of-fold,
#              test points = true held-out (shown as stars). Colored by
#              MDD/HC group. Dashed y=x line shows perfect prediction.
#              Gray band marks the empirical PHQ-9 dead zone (7–10).
# Right panel: per-test-subject residuals (pred - actual), labeled with
#              subject_id, group, and actual PHQ-9, sorted by actual.

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel 1: predicted vs actual
ax = axes[0]
train_is_mdd = (grp_train == "MDD")
test_is_mdd = (grp_test == "MDD")

ax.scatter(y_train[~train_is_mdd], loso_pred_train[~train_is_mdd],
           s=45, alpha=0.5, c="tab:blue", marker="o", label="Train HC (LOSO-OOF)")
ax.scatter(y_train[train_is_mdd], loso_pred_train[train_is_mdd],
           s=45, alpha=0.5, c="tab:red", marker="o", label="Train MDD (LOSO-OOF)")
ax.scatter(y_test[~test_is_mdd], y_pred_test[~test_is_mdd],
           s=200, alpha=0.95, c="tab:blue", marker="*",
           edgecolors="black", linewidths=1.2, label="Test HC")
ax.scatter(y_test[test_is_mdd], y_pred_test[test_is_mdd],
           s=200, alpha=0.95, c="tab:red", marker="*",
           edgecolors="black", linewidths=1.2, label="Test MDD")

lims = [-2, 28]
ax.plot(lims, lims, "k--", alpha=0.4, label="y = x (perfect)")
ax.axvspan(7, 11, color="gray", alpha=0.12, label="PHQ-9 dead zone")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual PHQ-9")
ax.set_ylabel("Predicted PHQ-9")
ax.set_title("LOSO Elastic Net — predicted vs actual PHQ-9")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

# --- Panel 2: per-test-subject residuals
ax = axes[1]
residuals_df = test_results.sort_values("PHQ9_actual")
colors = ["tab:red" if g == "MDD" else "tab:blue" for g in residuals_df["group"]]
resid = residuals_df["PHQ9_pred"] - residuals_df["PHQ9_actual"]
ax.bar(range(len(residuals_df)), resid, color=colors, edgecolor="black")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(residuals_df)))
ax.set_xticklabels(
    [f"{sid}\n({g}, PHQ={p})" for sid, g, p in
     zip(residuals_df.index, residuals_df["group"], residuals_df["PHQ9_actual"])],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylabel("Residual  (predicted − actual)")
rmse_test = float(np.sqrt(((y_test - y_pred_test) ** 2).mean()))
ax.set_title(f"Per-test-subject residuals   (test RMSE = {rmse_test:.2f})")
ax.grid(alpha=0.3, axis="y")
ax.legend(handles=[Patch(facecolor="tab:red", label="MDD"),
                   Patch(facecolor="tab:blue", label="HC")],
          loc="best")

plt.tight_layout()
plt.show()


In [ ]:
# CELL 9 — Helper: fit LOSO Elastic Net on a subset of audio files.
#
# For each grouping below we pick a subset of the 29 audio files
# (by valence or by task), compute the per-subject 88-dim mean over
# ONLY those files, and run the same LOSO Elastic Net pipeline as
# Cell 6 on the SAME 80/20 split from Cell 5. Reusing idx_train /
# idx_test lets us compare groupings like-for-like on identical test
# subjects (differences in RMSE reflect the grouping, not split noise).

def fit_subset_loso_enet(file_numbers, label):
    """Per-subject mean over `file_numbers`, LOSO ElasticNetCV on the
    training subjects from Cell 5, predict the same test subjects.
    Returns a dict of predictions + chosen hyperparameters + metrics."""

    frames = [func_dfs[fn] for fn in file_numbers if fn in func_dfs]
    stacked_sub = pd.concat(frames, axis=0)
    subj_mean = stacked_sub.groupby(level=0).mean()
    subj_mean = subj_mean.loc[subject_info.index]

    X_tr = subj_mean.loc[idx_train].values
    X_te = subj_mean.loc[idx_test].values

    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    X_te_s = scaler.transform(X_te)

    loo_local = LeaveOneOut()
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
        alphas=np.logspace(-3, 1, 40),
        cv=loo_local, max_iter=20000, n_jobs=-1,
        random_state=RANDOM_STATE,
    ).fit(X_tr_s, y_train)

    final = ElasticNet(
        alpha=enet.alpha_, l1_ratio=enet.l1_ratio_,
        max_iter=20000, random_state=RANDOM_STATE,
    )
    loso_train = cross_val_predict(final, X_tr_s, y_train, cv=loo_local)
    y_pred_te = enet.predict(X_te_s)

    return {
        "label": label,
        "n_files": len(file_numbers),
        "file_numbers": file_numbers,
        "alpha": float(enet.alpha_),
        "l1_ratio": float(enet.l1_ratio_),
        "n_nonzero": int((enet.coef_ != 0).sum()),
        "coef": enet.coef_,
        "train_loso_pred": loso_train,
        "test_pred": y_pred_te,
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, y_pred_te))),
        "test_mae": float(mean_absolute_error(y_test, y_pred_te)),
        "test_r2": float(r2_score(y_test, y_pred_te)),
        "train_loso_rmse": float(np.sqrt(mean_squared_error(y_train, loso_train))),
        "train_loso_mae": float(mean_absolute_error(y_train, loso_train)),
    }

print("Helper defined: fit_subset_loso_enet(file_numbers, label)")


In [ ]:
# CELL 10 — Grouping 1: 3 models by valence.
# Positive / Neutral / Negative. Files with no valence (19 = passage
# reading, 29 = TAT) are excluded per the user spec.

VALENCES = ["positive", "neutral", "negative"]

valence_file_groups = {
    v: audio_file_map.loc[audio_file_map["valence"] == v, "file_number"].tolist()
    for v in VALENCES
}
print("Valence → file numbers:")
for v, files in valence_file_groups.items():
    print(f"  {v:9s} ({len(files)} files): {files}")

valence_results = {
    v: fit_subset_loso_enet(files, f"valence={v}")
    for v, files in valence_file_groups.items()
}

valence_summary = pd.DataFrame({
    v: {
        "n_files": r["n_files"],
        "alpha": round(r["alpha"], 4),
        "l1_ratio": r["l1_ratio"],
        "n_nonzero": r["n_nonzero"],
        "train_LOSO_RMSE": round(r["train_loso_rmse"], 3),
        "train_LOSO_MAE": round(r["train_loso_mae"], 3),
        "test_RMSE": round(r["test_rmse"], 3),
        "test_MAE": round(r["test_mae"], 3),
        "test_R2": round(r["test_r2"], 3),
    } for v, r in valence_results.items()
}).T
print("\nValence model summary:")
print(valence_summary)


In [ ]:
# CELL 11 — Valence models — visualization.
# Top row: predicted-vs-actual PHQ-9 for each valence (same style as Cell 8
# left panel). Bottom: grouped bar chart of train LOSO-OOF RMSE and test
# RMSE per valence, with the mean-predictor baseline as a horizontal
# reference.

fig, axes = plt.subplots(
    2, 3, figsize=(16, 10),
    gridspec_kw={"height_ratios": [3, 2]},
)

for col, v in enumerate(VALENCES):
    r = valence_results[v]
    ax = axes[0, col]
    train_is_mdd = (grp_train == "MDD")
    test_is_mdd = (grp_test == "MDD")

    ax.scatter(y_train[~train_is_mdd], r["train_loso_pred"][~train_is_mdd],
               s=40, alpha=0.5, c="tab:blue", marker="o",
               label="Train HC (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_train[train_is_mdd], r["train_loso_pred"][train_is_mdd],
               s=40, alpha=0.5, c="tab:red", marker="o",
               label="Train MDD (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_test[~test_is_mdd], r["test_pred"][~test_is_mdd],
               s=180, alpha=0.95, c="tab:blue", marker="*",
               edgecolors="black", linewidths=1.1,
               label="Test HC" if col == 0 else None)
    ax.scatter(y_test[test_is_mdd], r["test_pred"][test_is_mdd],
               s=180, alpha=0.95, c="tab:red", marker="*",
               edgecolors="black", linewidths=1.1,
               label="Test MDD" if col == 0 else None)

    lims = [-2, 28]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.axvspan(7, 11, color="gray", alpha=0.12)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual PHQ-9")
    if col == 0:
        ax.set_ylabel("Predicted PHQ-9")
        ax.legend(loc="lower right", fontsize=8)
    ax.set_title(f"{v.capitalize()}\n(test RMSE = {r['test_rmse']:.2f},  n_files = {r['n_files']})")
    ax.grid(alpha=0.3)

# Merge bottom row into a single wide axes for the bar chart
gs = axes[1, 0].get_gridspec()
for ax in axes[1, :]:
    ax.remove()
ax_bar = fig.add_subplot(gs[1, :])

xpos = np.arange(len(VALENCES))
train_rmse = [valence_results[v]["train_loso_rmse"] for v in VALENCES]
test_rmse = [valence_results[v]["test_rmse"] for v in VALENCES]
w = 0.35
ax_bar.bar(xpos - w/2, train_rmse, w, label="Train LOSO-OOF RMSE",
           color="tab:gray", edgecolor="black")
ax_bar.bar(xpos + w/2, test_rmse, w, label="Test RMSE",
           color="tab:orange", edgecolor="black")
for i, (tr, te) in enumerate(zip(train_rmse, test_rmse)):
    ax_bar.text(i - w/2, tr + 0.05, f"{tr:.2f}", ha="center", fontsize=9)
    ax_bar.text(i + w/2, te + 0.05, f"{te:.2f}", ha="center", fontsize=9)

baseline_rmse = float(np.sqrt(((y_test - y_train.mean()) ** 2).mean()))
ax_bar.axhline(baseline_rmse, color="black", linestyle="--", alpha=0.6,
               label=f"Mean-predictor baseline ({baseline_rmse:.2f})")

ax_bar.set_xticks(xpos)
ax_bar.set_xticklabels([v.capitalize() for v in VALENCES])
ax_bar.set_ylabel("RMSE")
ax_bar.set_title("RMSE comparison across valence models")
ax_bar.legend(loc="best")
ax_bar.grid(alpha=0.3, axis="y")

fig.suptitle("Grouping 1 — PHQ-9 prediction by valence",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 12 — Grouping 2: 4 models by task.
# Tasks (from audio_file_map): Interview, Passage Reading, Word Reading,
# Picture Description.
#
# NOTE: Passage Reading has only 1 file per subject (file 19). "Per-subject
# mean over 1 file" is just that single file — noisier than tasks with 6+
# files averaged. Flag when interpreting its RMSE.

TASKS = audio_file_map["task"].unique().tolist()

task_file_groups = {
    t: audio_file_map.loc[audio_file_map["task"] == t, "file_number"].tolist()
    for t in TASKS
}
print("Task → file numbers:")
for t, files in task_file_groups.items():
    print(f"  {t:22s} ({len(files):2d} files): {files}")

task_results = {
    t: fit_subset_loso_enet(files, f"task={t}")
    for t, files in task_file_groups.items()
}

task_summary = pd.DataFrame({
    t: {
        "n_files": r["n_files"],
        "alpha": round(r["alpha"], 4),
        "l1_ratio": r["l1_ratio"],
        "n_nonzero": r["n_nonzero"],
        "train_LOSO_RMSE": round(r["train_loso_rmse"], 3),
        "train_LOSO_MAE": round(r["train_loso_mae"], 3),
        "test_RMSE": round(r["test_rmse"], 3),
        "test_MAE": round(r["test_mae"], 3),
        "test_R2": round(r["test_r2"], 3),
    } for t, r in task_results.items()
}).T
print("\nTask model summary:")
print(task_summary)


In [ ]:
# CELL 13 — Task models — visualization.
# Top row: predicted-vs-actual PHQ-9 for each task (same style as Cell 8
# and Cell 11). Bottom: bar chart of train LOSO-OOF RMSE and test RMSE
# per task, with the mean-predictor baseline as a horizontal reference.

n_tasks = len(TASKS)
fig, axes = plt.subplots(
    2, n_tasks, figsize=(5 * n_tasks, 10),
    gridspec_kw={"height_ratios": [3, 2]},
)

for col, t in enumerate(TASKS):
    r = task_results[t]
    ax = axes[0, col]
    train_is_mdd = (grp_train == "MDD")
    test_is_mdd = (grp_test == "MDD")

    ax.scatter(y_train[~train_is_mdd], r["train_loso_pred"][~train_is_mdd],
               s=35, alpha=0.5, c="tab:blue", marker="o",
               label="Train HC (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_train[train_is_mdd], r["train_loso_pred"][train_is_mdd],
               s=35, alpha=0.5, c="tab:red", marker="o",
               label="Train MDD (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_test[~test_is_mdd], r["test_pred"][~test_is_mdd],
               s=170, alpha=0.95, c="tab:blue", marker="*",
               edgecolors="black", linewidths=1.0,
               label="Test HC" if col == 0 else None)
    ax.scatter(y_test[test_is_mdd], r["test_pred"][test_is_mdd],
               s=170, alpha=0.95, c="tab:red", marker="*",
               edgecolors="black", linewidths=1.0,
               label="Test MDD" if col == 0 else None)

    lims = [-2, 28]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.axvspan(7, 11, color="gray", alpha=0.12)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual PHQ-9")
    if col == 0:
        ax.set_ylabel("Predicted PHQ-9")
        ax.legend(loc="lower right", fontsize=7)
    ax.set_title(f"{t}\n(test RMSE = {r['test_rmse']:.2f},  n_files = {r['n_files']})",
                 fontsize=10)
    ax.grid(alpha=0.3)

gs = axes[1, 0].get_gridspec()
for ax in axes[1, :]:
    ax.remove()
ax_bar = fig.add_subplot(gs[1, :])

xpos = np.arange(n_tasks)
train_rmse = [task_results[t]["train_loso_rmse"] for t in TASKS]
test_rmse = [task_results[t]["test_rmse"] for t in TASKS]
w = 0.35
ax_bar.bar(xpos - w/2, train_rmse, w, label="Train LOSO-OOF RMSE",
           color="tab:gray", edgecolor="black")
ax_bar.bar(xpos + w/2, test_rmse, w, label="Test RMSE",
           color="tab:orange", edgecolor="black")
for i, (tr, te) in enumerate(zip(train_rmse, test_rmse)):
    ax_bar.text(i - w/2, tr + 0.05, f"{tr:.2f}", ha="center", fontsize=9)
    ax_bar.text(i + w/2, te + 0.05, f"{te:.2f}", ha="center", fontsize=9)

baseline_rmse = float(np.sqrt(((y_test - y_train.mean()) ** 2).mean()))
ax_bar.axhline(baseline_rmse, color="black", linestyle="--", alpha=0.6,
               label=f"Mean-predictor baseline ({baseline_rmse:.2f})")

ax_bar.set_xticks(xpos)
ax_bar.set_xticklabels(TASKS, rotation=15, ha="right")
ax_bar.set_ylabel("RMSE")
ax_bar.set_title("RMSE comparison across task models")
ax_bar.legend(loc="best")
ax_bar.grid(alpha=0.3, axis="y")

fig.suptitle("Grouping 2 — PHQ-9 prediction by task",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 14 — Bootstrap prediction intervals for the overall model.
#
# Method: resample the training set (with replacement) B times, refit
# ElasticNet at the hyperparameters chosen in Cell 6, predict each test
# point -> distribution of B predictions per test subject -> take the
# 2.5th and 97.5th percentiles.
#
# Add residual noise: each bootstrap prediction is perturbed by
# a random draw from the empirical residual distribution (the LOSO
# out-of-fold residuals from Cell 6), so the interval captures both
# model uncertainty AND irreducible noise.

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

B = 500          # number of bootstrap iterations
CI_LEVEL = 0.95  # 95% prediction interval
rng = np.random.RandomState(RANDOM_STATE)

# LOSO residuals from Cell 6 (estimate irreducible noise)
loso_residuals = y_train - loso_pred_train

boot_preds_test = np.zeros((B, len(y_test)))
boot_preds_train = np.zeros((B, len(y_train)))

for b in range(B):
    # Resample training indices with replacement
    idx_boot = rng.choice(len(y_train), size=len(y_train), replace=True)
    X_boot = X_train[idx_boot]
    y_boot = y_train[idx_boot]

    # Refit scaler + model on bootstrap sample
    sc_b = StandardScaler().fit(X_boot)
    X_boot_s = sc_b.transform(X_boot)
    X_test_s_b = sc_b.transform(X_test)
    X_train_s_b = sc_b.transform(X_train)

    enet_b = ElasticNet(
        alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_,
        max_iter=20000, random_state=RANDOM_STATE,
    ).fit(X_boot_s, y_boot)

    pred_test_b = enet_b.predict(X_test_s_b)
    pred_train_b = enet_b.predict(X_train_s_b)

    # Add residual noise to capture irreducible error
    noise_test = rng.choice(loso_residuals, size=len(y_test), replace=True)
    noise_train = rng.choice(loso_residuals, size=len(y_train), replace=True)

    boot_preds_test[b] = pred_test_b + noise_test
    boot_preds_train[b] = pred_train_b + noise_train

alpha_half = (1 - CI_LEVEL) / 2

# Test intervals
test_lo = np.percentile(boot_preds_test, 100 * alpha_half, axis=0)
test_hi = np.percentile(boot_preds_test, 100 * (1 - alpha_half), axis=0)
test_median = np.median(boot_preds_test, axis=0)
test_width = test_hi - test_lo

# Train intervals (for comparison)
train_lo = np.percentile(boot_preds_train, 100 * alpha_half, axis=0)
train_hi = np.percentile(boot_preds_train, 100 * (1 - alpha_half), axis=0)

test_coverage = np.mean((y_test >= test_lo) & (y_test <= test_hi))
train_coverage = np.mean((y_train >= train_lo) & (y_train <= train_hi))

print(f"Bootstrap prediction intervals ({int(CI_LEVEL*100)}%, B={B}):")
print(f"  Test  — coverage: {test_coverage:.0%}  mean width: {test_width.mean():.2f} PHQ-9 points")
print(f"  Train — coverage: {train_coverage:.0%}")
print()

pi_df = pd.DataFrame({
    "subject_id": idx_test,
    "group": grp_test,
    "PHQ9_actual": y_test,
    "PHQ9_pred": y_pred_test,
    "PI_low": test_lo,
    "PI_high": test_hi,
    "PI_width": test_width,
    "covered": (y_test >= test_lo) & (y_test <= test_hi),
}).set_index("subject_id").sort_values("PHQ9_actual")

print("Per-test-subject prediction intervals:")
print(pi_df.round(2))


In [ ]:
# CELL 15 — Prediction interval visualization.
#
# Left:  Predicted vs actual with vertical error bars showing the 95% PI
#        for each test subject. Green = actual falls inside PI, red = missed.
# Right: Interval width vs actual PHQ-9 — are intervals wider for certain
#        severity levels?

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Panel 1: predicted vs actual with PIs ---
ax = axes[0]

order = np.argsort(y_test)
y_act_sorted = y_test[order]
y_pred_sorted = y_pred_test[order]
lo_sorted = test_lo[order]
hi_sorted = test_hi[order]
grp_sorted = grp_test[order]
covered_sorted = ((y_act_sorted >= lo_sorted) & (y_act_sorted <= hi_sorted))

for i in range(len(y_test)):
    color = "tab:green" if covered_sorted[i] else "tab:red"
    ax.plot([y_act_sorted[i], y_act_sorted[i]], [lo_sorted[i], hi_sorted[i]],
            color=color, linewidth=2.5, alpha=0.6, zorder=1)

for i in range(len(y_test)):
    marker_color = "tab:red" if grp_sorted[i] == "MDD" else "tab:blue"
    ax.scatter(y_act_sorted[i], y_pred_sorted[i],
               s=120, c=marker_color, marker="*", edgecolors="black",
               linewidths=0.8, zorder=3)

lims = [-2, 28]
ax.plot(lims, lims, "k--", alpha=0.4, label="y = x (perfect)")
ax.axvspan(7, 11, color="gray", alpha=0.08)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual PHQ-9")
ax.set_ylabel("Predicted PHQ-9")
ax.set_title(f"Predicted vs Actual with {int(CI_LEVEL*100)}% Prediction Intervals\n"
             f"(coverage = {test_coverage:.0%}, mean width = {test_width.mean():.1f})")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="*", color="w", markerfacecolor="tab:red",
           markeredgecolor="black", markersize=12, label="MDD"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="tab:blue",
           markeredgecolor="black", markersize=12, label="HC"),
    Line2D([0], [0], color="tab:green", linewidth=2.5, label="PI covers actual"),
    Line2D([0], [0], color="tab:red", linewidth=2.5, label="PI misses actual"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
ax.grid(alpha=0.3)

# --- Panel 2: interval width vs actual ---
ax = axes[1]
colors = ["tab:red" if g == "MDD" else "tab:blue" for g in grp_test]
ax.scatter(y_test, test_width, s=80, c=colors, edgecolors="black",
           linewidths=0.8, alpha=0.8)
ax.axhline(test_width.mean(), color="gray", linestyle="--", alpha=0.6,
           label=f"Mean width = {test_width.mean():.1f}")
ax.set_xlabel("Actual PHQ-9")
ax.set_ylabel(f"{int(CI_LEVEL*100)}% PI Width (PHQ-9 points)")
ax.set_title("Prediction Interval Width by Depression Severity")
ax.legend(handles=[
    Patch(facecolor="tab:red", edgecolor="black", label="MDD"),
    Patch(facecolor="tab:blue", edgecolor="black", label="HC"),
    Line2D([0], [0], color="gray", linestyle="--", label=f"Mean = {test_width.mean():.1f}"),
], loc="best", fontsize=8)
ax.grid(alpha=0.3)

fig.suptitle("Bootstrap Prediction Intervals — Overall Model",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 16 — Conformal prediction intervals (simpler alternative).
#
# Split conformal using LOSO residuals: the interval is
#   y_hat +/- q_{1-alpha}(|LOSO residuals|)
# where q is the (1-alpha) quantile of the absolute LOSO residuals.
# This is distribution-free and has finite-sample coverage guarantees
# under exchangeability. Simpler than bootstrap but gives constant-width
# intervals (same width for every test point).

import numpy as np

# Absolute LOSO residuals from training
abs_resid = np.abs(y_train - loso_pred_train)

# Conformal quantile with finite-sample correction
n_cal = len(abs_resid)
q_level = np.ceil((1 + n_cal) * CI_LEVEL) / n_cal
q_level = min(q_level, 1.0)
conformal_radius = np.quantile(abs_resid, q_level)

conf_lo = y_pred_test - conformal_radius
conf_hi = y_pred_test + conformal_radius
conf_coverage = np.mean((y_test >= conf_lo) & (y_test <= conf_hi))

print(f"Conformal prediction intervals ({int(CI_LEVEL*100)}%):")
print(f"  Radius (constant): +/-{conformal_radius:.2f} PHQ-9 points")
print(f"  Interval width:    {2*conformal_radius:.2f} PHQ-9 points")
print(f"  Test coverage:     {conf_coverage:.0%}")
print()
print("Comparison:")
print(f"  Bootstrap — mean width: {test_width.mean():.2f}, coverage: {test_coverage:.0%}")
print(f"  Conformal — fixed width: {2*conformal_radius:.2f}, coverage: {conf_coverage:.0%}")
print()
print("The conformal interval is wider but comes with a formal coverage guarantee.")
print("The bootstrap intervals are adaptive (narrower for easier subjects) but heuristic.")


In [ ]:
# CELL 17 — Conformal prediction intervals for each valence and task model.
#
# For each grouping, compute conformal PI radius from the LOSO residuals.
# This let's us compare: does a particular grouping give tighter intervals?

def compute_conformal_pi(result_dict, ci_level=0.95):
    abs_resid = np.abs(y_train - result_dict["train_loso_pred"])
    n_cal = len(abs_resid)
    q_level = min(np.ceil((1 + n_cal) * ci_level) / n_cal, 1.0)
    radius = float(np.quantile(abs_resid, q_level))

    pred = result_dict["test_pred"]
    lo = pred - radius
    hi = pred + radius
    coverage = float(np.mean((y_test >= lo) & (y_test <= hi)))

    return {
        "pi_radius": radius,
        "pi_width": 2 * radius,
        "pi_coverage": coverage,
        "pi_lo": lo,
        "pi_hi": hi,
    }

# Compute for all valence and task models
valence_pi = {v: compute_conformal_pi(r) for v, r in valence_results.items()}
task_pi = {t: compute_conformal_pi(r) for t, r in task_results.items()}

overall_pi = {
    "pi_radius": conformal_radius,
    "pi_width": 2 * conformal_radius,
    "pi_coverage": conf_coverage,
}

# Summary table
pi_rows = [{"Model": "Overall (all 29 files)", "Test RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred_test))), **overall_pi}]
for v in VALENCES:
    pi_rows.append({
        "Model": f"Valence: {v}",
        "Test RMSE": valence_results[v]["test_rmse"],
        **valence_pi[v],
    })
for t in TASKS:
    pi_rows.append({
        "Model": f"Task: {t}",
        "Test RMSE": task_results[t]["test_rmse"],
        **task_pi[t],
    })

pi_summary = pd.DataFrame(pi_rows).set_index("Model")
pi_summary = pi_summary[["Test RMSE", "pi_width", "pi_coverage"]].copy()
pi_summary.columns = ["Test RMSE", "95% PI Width", "Coverage"]
print("Conformal prediction intervals across all models:")
print(pi_summary.round(3))


In [ ]:
# CELL 18 — Grouping 3: valence x task cross-grouping
#
# Each audio file has BOTH a task and a valence. By crossing them we get
# fine-grained subsets like "positive interview questions only" or
# "negative word reading only." Some combinations may have very few files
# (or none — e.g. Passage Reading has no valence).
#
# For each valid combination, we run the same LOSO Elastic Net pipeline
# to see which specific task+valence combo gives the best signal.

# Build the cross-grouping
cross_groups = {}
for _, row in audio_file_map.iterrows():
    task = row["task"]
    valence = row["valence"] if pd.notna(row["valence"]) else "none"
    key = (task, valence)
    if key not in cross_groups:
        cross_groups[key] = []
    cross_groups[key].append(row["file_number"])

print("Valence x Task cross-grouping:")
print(f"{'Task':<25s} {'Valence':<10s} {'Files':<6s} File numbers")
print("-" * 65)
for (task, valence), files in sorted(cross_groups.items()):
    print(f"{task:<25s} {valence:<10s} {len(files):<6d} {files}")

# Fit models for all valid groups
cross_results = {}
for (task, valence), files in sorted(cross_groups.items()):
    label = f"{task} x {valence}"
    print(f"\nFitting: {label} ({len(files)} files)...")
    cross_results[(task, valence)] = fit_subset_loso_enet(files, label)

# Summary table
cross_rows = []
for (t, v), r in sorted(cross_results.items()):
    cross_rows.append({
        "Task": t,
        "Valence": v,
        "n_files": r["n_files"],
        "alpha": round(r["alpha"], 4),
        "l1_ratio": r["l1_ratio"],
        "n_nonzero": r["n_nonzero"],
        "train_LOSO_RMSE": round(r["train_loso_rmse"], 3),
        "test_RMSE": round(r["test_rmse"], 3),
        "test_MAE": round(r["test_mae"], 3),
        "test_R2": round(r["test_r2"], 3),
    })

cross_summary = pd.DataFrame(cross_rows)
print("\nValence x Task cross-grouping summary:")
print(cross_summary.to_string(index=False))


In [ ]:
# CELL 19 — Valence x Task heatmap of test RMSE.
#
# Rows = tasks, columns = valences. Cell color = test RMSE (lower = better).
# Cells with no valid combination are left blank.

import matplotlib.pyplot as plt
import numpy as np

all_tasks = sorted(set(t for t, v in cross_results.keys()))
all_valences = ["positive", "neutral", "negative", "none"]
all_valences = [v for v in all_valences if any(v == vv for _, vv in cross_results.keys())]

# Build the RMSE matrix
rmse_matrix = np.full((len(all_tasks), len(all_valences)), np.nan)
n_files_matrix = np.full((len(all_tasks), len(all_valences)), 0)

for i, t in enumerate(all_tasks):
    for j, v in enumerate(all_valences):
        if (t, v) in cross_results:
            rmse_matrix[i, j] = cross_results[(t, v)]["test_rmse"]
            n_files_matrix[i, j] = cross_results[(t, v)]["n_files"]

fig, ax = plt.subplots(figsize=(10, 6))

masked = np.ma.masked_invalid(rmse_matrix)
cmap = plt.cm.RdYlGn_r  # red = high RMSE (bad), green = low (good)
im = ax.imshow(masked, cmap=cmap, aspect="auto",
               vmin=np.nanmin(rmse_matrix) - 0.5,
               vmax=np.nanmax(rmse_matrix) + 0.5)

for i in range(len(all_tasks)):
    for j in range(len(all_valences)):
        if not np.isnan(rmse_matrix[i, j]):
            text = f"{rmse_matrix[i, j]:.2f}\n({int(n_files_matrix[i, j])} files)"
            color = "white" if rmse_matrix[i, j] > np.nanmedian(rmse_matrix) else "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=10,
                    fontweight="bold", color=color)
        else:
            ax.text(j, i, "N/A", ha="center", va="center", fontsize=10,
                    color="gray", style="italic")

ax.set_xticks(range(len(all_valences)))
ax.set_xticklabels([v.capitalize() for v in all_valences])
ax.set_yticks(range(len(all_tasks)))
ax.set_yticklabels(all_tasks)
ax.set_xlabel("Valence")
ax.set_ylabel("Task")

baseline_rmse = float(np.sqrt(((y_test - y_train.mean()) ** 2).mean()))
ax.set_title(f"Test RMSE by Task x Valence\n"
             f"(mean-predictor baseline = {baseline_rmse:.2f})",
             fontsize=13, fontweight="bold")

plt.colorbar(im, ax=ax, label="Test RMSE (lower = better)", shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# CELL 20 — Prediction intervals for the cross-grouped models.
#
# For each task x valence model, compute conformal PI and compare.
# Which combination gives the tightest prediction interval?

cross_pi = {
    (t, v): compute_conformal_pi(r)
    for (t, v), r in cross_results.items()
}

# Build PI width matrix for heatmap
pi_width_matrix = np.full((len(all_tasks), len(all_valences)), np.nan)
pi_coverage_matrix = np.full((len(all_tasks), len(all_valences)), np.nan)

for i, t in enumerate(all_tasks):
    for j, v in enumerate(all_valences):
        if (t, v) in cross_pi:
            pi_width_matrix[i, j] = cross_pi[(t, v)]["pi_width"]
            pi_coverage_matrix[i, j] = cross_pi[(t, v)]["pi_coverage"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: PI width heatmap ---
ax = axes[0]
masked_pw = np.ma.masked_invalid(pi_width_matrix)
im = ax.imshow(masked_pw, cmap="RdYlGn_r", aspect="auto",
               vmin=np.nanmin(pi_width_matrix) - 1,
               vmax=np.nanmax(pi_width_matrix) + 1)
for i in range(len(all_tasks)):
    for j in range(len(all_valences)):
        if not np.isnan(pi_width_matrix[i, j]):
            text = f"{pi_width_matrix[i, j]:.1f}"
            color = "white" if pi_width_matrix[i, j] > np.nanmedian(pi_width_matrix) else "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=11,
                    fontweight="bold", color=color)
        else:
            ax.text(j, i, "N/A", ha="center", va="center", fontsize=10,
                    color="gray", style="italic")
ax.set_xticks(range(len(all_valences)))
ax.set_xticklabels([v.capitalize() for v in all_valences])
ax.set_yticks(range(len(all_tasks)))
ax.set_yticklabels(all_tasks)
ax.set_title("95% Conformal PI Width\n(PHQ-9 points, lower = better)")
plt.colorbar(im, ax=ax, label="PI Width", shrink=0.8)

# --- Panel 2: bar chart comparing RMSE and PI width ---
ax = axes[1]
sorted_keys = sorted(cross_results.keys())
labels = [f"{t}\n{v}" for (t, v) in sorted_keys]
rmses = [cross_results[k]["test_rmse"] for k in sorted_keys]
widths = [cross_pi[k]["pi_width"] for k in sorted_keys]
coverages = [cross_pi[k]["pi_coverage"] for k in sorted_keys]

x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, rmses, w, label="Test RMSE", color="tab:orange", edgecolor="black")
ax.bar(x + w/2, widths, w, label="95% PI Width", color="tab:purple",
       edgecolor="black", alpha=0.7)

for i, (r, pw) in enumerate(zip(rmses, widths)):
    ax.text(i - w/2, r + 0.1, f"{r:.1f}", ha="center", fontsize=7)
    ax.text(i + w/2, pw + 0.1, f"{pw:.1f}", ha="center", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("PHQ-9 points")
ax.set_title("RMSE and PI Width by Task x Valence")
ax.legend(loc="best")
ax.grid(alpha=0.3, axis="y")

fig.suptitle("Prediction Intervals — Task x Valence Cross-Grouping",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print the best combination
best_key = min(cross_results.keys(), key=lambda k: cross_results[k]["test_rmse"])
print(f"\nBest test RMSE: {best_key[0]} x {best_key[1]} "
      f"(RMSE={cross_results[best_key]['test_rmse']:.3f}, "
      f"PI width={cross_pi[best_key]['pi_width']:.1f})")

tightest_key = min(cross_pi.keys(), key=lambda k: cross_pi[k]["pi_width"])
print(f"Tightest PI:    {tightest_key[0]} x {tightest_key[1]} "
      f"(PI width={cross_pi[tightest_key]['pi_width']:.1f}, "
      f"RMSE={cross_results[tightest_key]['test_rmse']:.3f})")


In [ ]:
# CELL 21 — Normalized error metrics for more generalizable reporting.
#
# Your notes mention wanting to present error "as percentage or out of 1"
# in addition to raw PHQ-9 points. Here are several normalizations:
#
#   NRMSE = RMSE / PHQ-9 range  (0-27)
#   NRMSE_IQR = RMSE / IQR(y_train)
#   MAPE = mean(|error| / |actual|)  — only for non-zero actuals
#   R^2 is already scale-free

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

phq9_range = 27  # theoretical max (0-27)

def normalized_metrics(y_true, y_pred, label=""):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    nrmse = rmse / phq9_range
    # MAPE only on non-zero actuals to avoid division by zero
    nonzero = y_true != 0
    if nonzero.sum() > 0:
        mape = np.mean(np.abs(y_true[nonzero] - y_pred[nonzero]) / np.abs(y_true[nonzero]))
    else:
        mape = np.nan
    return {
        "RMSE": round(rmse, 3),
        "MAE": round(mae, 3),
        "R2": round(r2, 3),
        "NRMSE (0-1)": round(nrmse, 3),
        "MAPE": round(mape, 3) if not np.isnan(mape) else "N/A",
    }

# Overall model
print("=== Normalized metrics — Overall model ===")
for k, v in normalized_metrics(y_test, y_pred_test).items():
    print(f"  {k}: {v}")

# By valence
print("\n=== Normalized metrics — By valence ===")
for val in VALENCES:
    r = valence_results[val]
    m = normalized_metrics(y_test, r["test_pred"])
    print(f"  {val:10s}  RMSE={m['RMSE']}  NRMSE={m['NRMSE (0-1)']}  R2={m['R2']}")

# By task
print("\n=== Normalized metrics — By task ===")
for t in TASKS:
    r = task_results[t]
    m = normalized_metrics(y_test, r["test_pred"])
    print(f"  {t:22s}  RMSE={m['RMSE']}  NRMSE={m['NRMSE (0-1)']}  R2={m['R2']}")

print("\nNRMSE interpretation: fraction of the full PHQ-9 scale (0-27).")
print("E.g., NRMSE=0.25 means typical error is about 25% of the scale range.")
